In [0]:
!uv pip install -r ../requirements.txt --upgrade

In [0]:
!uv pip install unsloth==2026.7.2 --no-deps

In [0]:
!pip uninstall torchvision -y

In [0]:
%restart_python

In [0]:
%run ../utilities/config

In [0]:
from unsloth import FastLanguageModel
import torch

max_seq_length = instruct_max_seq_length
base_model_path =  stage1_merged_path   # or "meta-llama/Llama-3.2-1B

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model_path, #stage1_merged_path,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)


In [0]:
import json
from datasets import Dataset

rows = []
with open(instruction_dataset) as f:
    for line in f:
        rows.append(json.loads(line))

print(f"Loaded {len(rows)} instruction/response pairs")
rows[0]

In [0]:
ALPACA_PROMPT = (
    "Below is an instruction from an employee. Write an accurate, "
    "policy-grounded response for the HR Policy Assistant.\n\n"
    "### Instruction:\n{instruction}\n\n### Response:\n{response}"
)

In [0]:
ALPACA_PROMPT_WITH_INPUT = (
    "Below is an instruction from an employee, along with prior conversation "
    "context. Write an accurate, policy-grounded response for the HR Policy Assistant.\n\n"
    "### Context:\n{input}\n\n### Instruction:\n{instruction}\n\n### Response:\n{response}"
)

ALPACA_PROMPT = (
    "Below is an instruction from an employee. Write an accurate, "
    "policy-grounded response for the HR Policy Assistant.\n\n"
    "### Instruction:\n{instruction}\n\n### Response:\n{response}"
)

def format_example(ex):
    if ex.get("input"):
        text = ALPACA_PROMPT_WITH_INPUT.format(
            input=ex["input"], instruction=ex["instruction"], response=ex["output"]
        )
    else:
        text = ALPACA_PROMPT.format(instruction=ex["instruction"], response=ex["output"])
    return {"text": text + tokenizer.eos_token}


dataset = Dataset.from_list(rows).map(format_example)
dataset[0]["text"]

In [0]:
def token_length(example):
    return {
        "length": len(
            tokenizer(
                example["text"],
                add_special_tokens=True,
            )["input_ids"]
        )
    }

dataset = dataset.map(token_length)
lengths = dataset["length"]

print(max(lengths))
print(sum(lengths) / len(lengths))

In [0]:
dataset = dataset.filter(
        lambda x: x["length"] <= max_seq_length
    )
dataset=dataset.remove_columns("length")

In [0]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,  
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=64,  # Scaled with rank
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

In [0]:
model.print_trainable_parameters()


In [0]:
# Split dataset into train/eval to monitor memorization
# Use 10% for evaluation (100 examples)

train_test_split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = train_test_split["train"].shuffle(seed=42)
eval_dataset = train_test_split["test"]

print(f"Training set: {len(train_dataset)} examples")
print(f"Evaluation set: {len(eval_dataset)} examples")
print("\n⚠️  Evaluation will help monitor if the model is memorizing training data!")

In [0]:
from trl import SFTTrainer, SFTConfig


training_args = SFTConfig(
    output_dir="/Volumes/workspace/ai_model/sft_config/stage2_sft",
    
    # CRITICAL: Much lower learning rate for better memorization
    learning_rate=1e-4,  # Changed from 1e-4
    
    # More epochs for better retention
    num_train_epochs=3, 
    
    # Batch size
    per_device_train_batch_size=2,  
    gradient_accumulation_steps=8,
    
    # Warmup
    warmup_ratio=0.1,
    
    # Logging and evaluation
    logging_steps=5,
    eval_strategy="steps",  # Evaluate during training
    eval_steps=25,          # : Evaluate every 25 steps
    save_strategy="steps",  # : Save checkpoints
    save_steps=25,         # : Save every 25 steps
    save_total_limit=3,     # Keep only last 3 checkpoints
    load_best_model_at_end=True,  # : Load best checkpoint at end
    metric_for_best_model="eval_loss",  # Use eval loss to pick best
    
    # Optimization
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    
    # Other settings
    seed=42,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,
    report_to="mlflow",
)



In [0]:
from trl import SFTTrainer
import os

# FIX: Disable multiprocessing to prevent tokenization hang
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Create trainer with BOTH train and eval datasets
# dataset_num_proc=1 prevents the 0% hang during tokenization
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,  # 900 examples
    eval_dataset=eval_dataset,    # 100 examples for monitoring
    args=training_args,
    dataset_num_proc=1,  # CRITICAL: Disable multiprocessing to prevent hang
)



In [0]:
import mlflow

# Set experiment
mlflow.set_experiment("/Shared/LLM/HRPolicy")

# Enable autologging
mlflow.autolog()
mlflow.enable_system_metrics_logging()
mlflow.set_system_metrics_sampling_interval(5)

print("="*80)

print("="*80 + "\n")

with mlflow.start_run(run_name="Stage2_Instruction"):
    trainer_stats = trainer.train()
    print("\n" + "="*80)
    print("TRAINING COMPLETED!")
    print("="*80)
    print(trainer_stats)
    mlflow.log_param(
        "best_checkpoint",
        trainer.state.best_model_checkpoint,
    )

    mlflow.log_metric(
        "best_eval_loss",
        trainer.state.best_metric,
    )


In [0]:
mlflow.autolog(disable=True)

In [0]:
metrics = trainer.evaluate()
print(metrics)

In [0]:
print(trainer.state.best_model_checkpoint)
print(trainer.state.best_metric)

In [0]:
from unsloth import FastLanguageModel

best_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=trainer.state.best_model_checkpoint,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

trainer.model = best_model

metrics=trainer.evaluate()
print(metrics)

In [0]:
print("="*60)
print("Best checkpoint :", trainer.state.best_model_checkpoint)
print("Best eval_loss  :", trainer.state.best_metric)
print("Current eval    :", metrics["eval_loss"])
print("="*60)

assert abs(metrics["eval_loss"] - trainer.state.best_metric) < 1e-5

In [0]:
import shutil
import tempfile
import os

# Save adapter and tokenizer directly to Volume
best_model.save_pretrained(stage2_adapter_path)
tokenizer.save_pretrained(stage2_adapter_path)
print(f"✓ Adapter saved to {stage2_adapter_path}")

# Ensure merged directory exists
dbutils.fs.mkdirs(stage2_merged_path)

# Save merged model to temp dir first, then copy to Volume
with tempfile.TemporaryDirectory() as temp_dir:
    print(f"Saving merged model to temporary directory: {temp_dir}")
    best_model.save_pretrained_merged(temp_dir, tokenizer, save_method="merged_16bit")
    
    print(f"Copying merged model to Volume: {stage2_merged_path}")
    # Copy contents from temp dir to Volume
    for item in os.listdir(temp_dir):
        src = os.path.join(temp_dir, item)
        dst = os.path.join(stage2_merged_path, item)
        if os.path.isfile(src):
            shutil.copy2(src, dst)
            print(f"  Copied: {item}")
    
    print("✓ Merged model saved successfully to {}".format(stage2_merged_path))

In [0]:
FastLanguageModel.for_inference(best_model)

def ask(instruction):
    prompt = ALPACA_PROMPT.format(instruction=instruction, response="")
    inputs = tokenizer(prompt, return_tensors="pt").to(best_model.device)
    out = best_model.generate(**inputs, max_new_tokens=120, do_sample=True)
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    return text.split("### Response:")[-1].strip()

print(ask("Can I use my earned leave during my resignation notice period?"))

In [0]:
print(ask("I'm a security staff member and I want to know if I should search the femalie visiotors when female staff member not present"))

In [0]:
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM

# MODEL_PATH = stage2_merged_path

# tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_PATH,
#     torch_dtype=torch.float16,
#     device_map="auto"
# )

# #model.eval()

In [0]:
# # Restart Python to clear corrupted GPU memory
# dbutils.library.restartPython()

In [0]:
# # After restart, load model fresh for inference
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM

# MODEL_PATH = stage2_merged_path

# tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_PATH,
#     torch_dtype=torch.float16,
#     device_map="auto"
# )

# model.eval()

# def ask(instruction):
#     prompt = ALPACA_PROMPT.format(instruction=instruction, response="")
#     inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
#     with torch.no_grad():
#         out = model.generate(**inputs, max_new_tokens=120, do_sample=False)
    
#     text = tokenizer.decode(out[0], skip_special_tokens=True)
#     return text.split("### Response:")[-1].strip()

# print(ask("What discount do I get on medicines as an employee?"))

In [0]:
# FastLanguageModel.for_inference(model)

# def ask(instruction):
#     prompt = PROMPT_TEMPLATE.format(instruction=instruction)
#     inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
#     out = model.generate(**inputs, max_new_tokens=120, do_sample=False)
#     text = tokenizer.decode(out[0], skip_special_tokens=True)
#     return text.split("### Response:")[-1].strip()

# print(ask("why health checkup important prior to joining"))